<a href="https://colab.research.google.com/github/btwiamsujal/Python_Files/blob/main/Fuzzylogic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U scikit-fuzzy
import numpy as np
import skfuzzy as fuzz
import skfuzzy.control as ctrl
import yfinance as yf

# Function to get stock data with full details
def get_stock_data(ticker):
  # Identity if the ticker belongs to Indian stock
  if ticker.isalpha():
    ticker = ticker #".NS"   Append ".NS" for NSE stocks

  stock = yf.Ticker(ticker)
  hist = stock.history(period="1d", interval="5m")

  if hist.empty:
    print("Error: No data found for. Check the stock ticker!")
    return None, None, None, None, None, None, None, None

  latest_price = hist["Close"].iloc[-1] #latest closing price
  price_change = ((hist['Close'].iloc[-1] - hist['Close'].iloc[-2]) / hist['Close'].iloc[-2] * 100)  # Price in %
  volume_change = ((hist['Volume'].iloc[-1] - hist['Volume'].iloc[-2]) / hist['Volume'].iloc[-2] * 100)  # Volume in %

  # Fetch additional stock details
  info = stock.info
  stock_name = info.get("longName", "N/A")
  sector = info.get("sector", "N/A")
  market_cap = info.get("marketCap", "N/A")
  pe_ratio = info.get("trailingPE", "N/A")
  day_high = info.get("dayHigh", "N/A")
  day_low = info.get("dayLow", "N/A")

  return price_change, volume_change, stock_name, sector, market_cap, pe_ratio, day_high, day_low

# Fuzzy Logic Variables
price_change = ctrl.Antecedent(np.arange(-5, 6, 1), 'price_change') # -5% to +5%
volume_change = ctrl.Antecedent(np.arange(-50, 51, 10), 'volume_change') # -50% to +50%
decision = ctrl.Consequent(np.arange(0, 3, 1), 'decision')

# Membership Functions
price_change['decrease'] = fuzz.trimf(price_change.universe, [-5, -5, 0])
price_change['stable'] = fuzz.trimf(price_change.universe, [-2, 0, 2])
price_change['increase'] = fuzz.trimf(price_change.universe, [0, 5, 5])

volume_change['low'] = fuzz.trimf(volume_change.universe, [-50, -50, 0])
volume_change['medium'] = fuzz.trimf(volume_change.universe, [-10, 0, 10])
volume_change['high'] = fuzz.trimf(volume_change.universe, [0, 50, 50])

decision['SELL'] = fuzz.trimf(decision.universe, [0, 0, 1])
decision['HOLD'] = fuzz.trimf(decision.universe, [0, 1, 2])
decision['BUY'] = fuzz.trimf(decision.universe, [1, 2, 2])

# Fuzzy Rules
rule1 = ctrl.Rule(price_change['decrease'] & volume_change['low'], decision['SELL'])
rule2 = ctrl.Rule(price_change['decrease'] & volume_change['medium'], decision['HOLD'])
rule3 = ctrl.Rule(price_change['decrease'] & volume_change['high'], decision['HOLD'])
rule4 = ctrl.Rule(price_change['stable'] & volume_change['low'], decision['HOLD'])
rule5 = ctrl.Rule(price_change['stable'] & volume_change['medium'], decision['HOLD'])
rule6 = ctrl.Rule(price_change['stable'] & volume_change['high'], decision['BUY'])
rule7 = ctrl.Rule(price_change['increase'] & volume_change['low'], decision['BUY'])
rule8 = ctrl.Rule(price_change['increase'] & volume_change['medium'], decision['BUY'])
rule9 = ctrl.Rule(price_change['increase'] & volume_change['high'], decision['BUY'])

# Control System
decision_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5, rule6, rule7, rule8, rule9])
decision_simulation = ctrl.ControlSystemSimulation(decision_ctrl)

# Get user input
ticker = input("Enter stock Ticker symbol (e.g., APPL, TSLA for US | RELIANCE, TCS for India):").upper()

#fetch stock data
price_change_value, volume_change_value, stock_name, sector, market_cap, pe_ratio, day_high, day_low = get_stock_data(ticker)

if price_change_value is not None and volume_change_value is not None:
  # Provide input to the fuzzy logic system
  decision_simulation.input['price_change'] = price_change_value
  decision_simulation.input['volume_change'] = volume_change_value

  # Compute the decision
  decision_simulation.compute()

  # Get Fuzzy output decision
  decision_value = decision_simulation.output['decision']

  # Convert fuzzy output to BUY, SELL or HOLD
  if decision_value < 0.5:
    decision = 'SELL'
  elif decision_value < 1.5:
    decision = 'HOLD'
  else:
    decision = 'BUY'

  print(f"Stock: {stock_name} ({ticker})")
  print(f"Sector: {sector}")
  print(f"Market Cap: {market_cap}")
  print(f"P/E Ratio: {pe_ratio}")
  print(f"Day High: {day_high}")
  print(f"Day Low: {day_low}")
  print(f"Price Change: {price_change_value:.2f}%")
  print(f"Volume Change: {volume_change_value:.2f}%")
  print(f"Decision: {decision}")
else:
  print("Unable to fetch stock data. Please check the ticker symbol and try again.")


Enter stock Ticker symbol (e.g., APPL, TSLA for US | RELIANCE, TCS for India):GOOGL
Stock: Alphabet Inc. (GOOGL)
Sector: Communication Services
Market Cap: 2200601362432
P/E Ratio: 22.318012
Day High: 185.34
Day Low: 179.08
Price Change: 0.04%
Volume Change: 31.52%
Decision: BUY
